# 02_08_Profiling and Cleaning Infrabel Punctuality Benchmark

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PATH_PROCESSED = Path("../data/processed/")

PATH_RAW = Path("../data/raw/")

# Table of Contents
- [8. INFRABEL PUNCTUALITY BENCHMARK](#8-infrabel-punctuality-benchmark)

- [8.1. Data Profiling](#81-data-profiling)

- [8.2. Data Preparation](#82-data-preparation)
    - [8.2.1. Converting Date Columns](#821-converting-date-columns)
    - [8.2.2. Dropping Irrelevant Columns](#822-dropping-irrelevant-columns)
    - [8.2.3. Dropping Irrelevant Years](#823-dropping-irrelevant-years)
    - [8.2.4. Rounding `float64` Column](#824-rounding-float64-column)
    - [8.2.5. Renaming Columns](#825-renaming-columns)

- [8.3. Export to Gold Layer](#83-export-to-gold-layer)


## 8. INFRABEL PUNCTUALITY BENCHMARK

This table contains the **official monthly punctuality rates** published by Infrabel. It will be used as an **external benchmark** to compare these rates with the project's own results. It will be loaded into Power BI as a disconnected table and is therefore intentionally kept outside the data warehouse, as it is not part of the analytical model.  

<br>

**DESCRIPTION**

The `monthly_punctuality_agg.parquet` dataset is loaded from the `raw` directory as a DataFrame, under the `infrabel_benchmark` name.  

* It has **125 entries** and **11 columns**.   

* It contains no missing values or duplicates rows.  

<br>

**TRANSFORMATION PROCESS**

1) The DataFrame is sorted by month (**Section 8.2.**).  

2) **Date columns**: Two columns, named respectively `jaar` and `maand`, contain date values but are of type `object`. The `jaar` column is converted to `int32` and the `maand` column to `datetime64` (**Section 8.2.1.**).


5) **Domain-specific decision about *"neutralized"* rates**: Nine columns repeat the same pattern three times:  
    
    For each month, a float column containing the monthly ponctuality rate (`regelmaat`), followed by two integer columns from which this rate is calculated: the total train count (`tel`) and the number of trains that arrived at station at least six minutes late (`reg`).  
    
    According to the Infrabel documentation, **the first pattern contains the raw values** while the two other patterns exclude delays *"whose causes are external to the activities of Infrabel or SNCB/NMBS"* (see **NOTE** below). 

    Therefore, **only the three raw columns** (`regelmaat`, `tel`, and `reg`) **are kept** for this analysis while the `neutralized` columns are excluded (**Section 8.2.2.**).  

4) **Rows**: The 125 dataset entries match 12 months by year from 2016 to 2025 inclusive (12 months * 10 years) and the first five months of 2026. Since this analysis focuses on 2025 and 2026, and given that the project's date dimension starts in 2020, **the years before 2020 are excluded** (**Section 8.2.3**).

5) **Punctuality rate columns**: Three columns are of type `float64` and appear to be the result of an arithmetic calculation, given that they measure monthly punctuality averages. They are **rounded to two decimals** to improve the clarity of future comparisons between the benchmark and the project rates (**Section 8.2.4.**). 

6) The columns are renamed in English (**Section 8.2.5.**).

<br>

**NOTE - Domain-specific decision**: The following explanation is quoted from the Infrabel Open Data portal:  
> "According to the terms of the Management Contract, certain delays whose causes are external to the activities of Infrabel or SNCB/NMBS (e.g. level crossing accidents, cable theft, delays originating from foreign railway networks, vandalism, assaults, etc.), or that result from planned and communicated infrastructure works, may be neutralized.
>
> The punctuality figure "with neutralization" is not intended to reflect the reality as perceived by passengers; rather, it is a tool that enables Infrabel and the railway operators to assess their own performance.
>
> To obtain an overall picture of punctuality, it is therefore preferable to refer to the punctuality figure "without neutralization"."

<br>

## 8.1. Data Profiling

In [3]:
infrabel_benchmark = pd.read_parquet(PATH_RAW / "monthly_punctuality_agg.parquet")
infrabel_benchmark.head()

,jaar,maand,regelmaat,tel,reg,min_rt,regelmaatneutralisatie,reg_neutr,tel_ytd,reg_ytd,regelmaat_ytd
0,2016-01-01,2016-02-01,90.875745,102233,92905,203292,95.165944,97291,202266,182804,90.378017
1,2016-01-01,2016-04-01,90.567481,99369,89996,195684,96.484819,95876,406313,367793,90.519624
2,2017-01-01,2017-06-01,87.280144,106888,93292,273664,92.029040,98368,636751,568164,89.228600
3,2017-01-01,2017-10-01,83.560281,107514,89839,330432,89.917592,96674,1057998,942591,89.091945
4,2018-01-01,2018-02-01,87.254892,101945,88952,250288,90.771494,92537,216206,190165,87.955468


In [4]:
infrabel_benchmark.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   jaar                    125 non-null    object 
 1   maand                   125 non-null    object 
 2   regelmaat               125 non-null    float64
 3   tel                     125 non-null    int64  
 4   reg                     125 non-null    int64  
 5   min_rt                  125 non-null    int64  
 6   regelmaatneutralisatie  125 non-null    float64
 7   reg_neutr               125 non-null    int64  
 8   tel_ytd                 125 non-null    int64  
 9   reg_ytd                 125 non-null    int64  
 10  regelmaat_ytd           125 non-null    float64
dtypes: float64(3), int64(6), object(2)
memory usage: 10.9+ KB


In [5]:
infrabel_benchmark.nunique()

jaar                       11
maand                     125
regelmaat                 125
tel                       125
reg                       125
min_rt                    125
regelmaatneutralisatie    125
reg_neutr                 124
tel_ytd                   125
reg_ytd                   125
regelmaat_ytd             125
dtype: int64

In [6]:
infrabel_benchmark.duplicated().sum()

np.int64(0)

In [7]:
infrabel_benchmark["maand"].duplicated().sum()

np.int64(0)

In [8]:
infrabel_benchmark["jaar"].drop_duplicates()

0     2016-01-01
2     2017-01-01
4     2018-01-01
8     2019-01-01
11    2020-01-01
15    2021-01-01
16    2022-01-01
22    2023-01-01
27    2024-01-01
31    2025-01-01
78    2026-01-01
Name: jaar, dtype: object

## 8.2. Data Quality and Cleaning

### 8.2.1. Handling Date Columns

In [9]:
infrabel_benchmark = infrabel_benchmark.sort_values("maand", ascending=False)

In [10]:
infrabel_benchmark.head()

,jaar,maand,regelmaat,tel,reg,min_rt,regelmaatneutralisatie,reg_neutr,tel_ytd,reg_ytd,regelmaat_ytd
79,2026-01-01,2026-05-01,92.388205,109777,101421,184749,94.920612,104201,552560,513338,92.901766
124,2026-01-01,2026-04-01,93.397770,113704,106197,165290,95.387146,108459,442783,411917,93.029091
123,2026-01-01,2026-03-01,93.050200,112550,104728,181743,95.281208,107239,329079,305720,92.901704
78,2026-01-01,2026-02-01,93.251613,108663,101330,162212,94.972530,103200,216529,200992,92.824518
122,2026-01-01,2026-01-01,92.394267,107866,99662,178328,94.306825,101725,107866,99662,92.394267


### 8.2.1. Converting Date Columns

The year column (`jaar`) contains only the year values while the month column (`maand`) contains both the year and the month values. Both are kept to facilitate the filtering by years or months in future dashboards. 

The date format is explicitly specified to raise an error if any values do not match the expected format, preventing the silent creation of `NaN` values.  

In [11]:
infrabel_benchmark["maand"] = pd.to_datetime(
    infrabel_benchmark["maand"],
    format="%Y-%m-%d"
)

In [12]:
infrabel_benchmark["jaar"] = pd.to_datetime(
    infrabel_benchmark["jaar"],
    format="%Y-%m-%d"
).dt.year

### 8.2.2. Dropping Irrelevant Columns

In [13]:
infrabel_benchmark.columns

Index(['jaar', 'maand', 'regelmaat', 'tel', 'reg', 'min_rt',
       'regelmaatneutralisatie', 'reg_neutr', 'tel_ytd', 'reg_ytd',
       'regelmaat_ytd'],
      dtype='str')

In [14]:
infrabel_benchmark = (
    infrabel_benchmark.drop(columns=["regelmaatneutralisatie", "reg_neutr", 
                                     "tel_ytd", "reg_ytd", "regelmaat_ytd"])
)

### 8.2.3. Dropping Irrelevant Years

In [ ]:
infrabel_benchmark = (
    infrabel_benchmark.loc[infrabel_benchmark["jaar"] >= 2020].reset_index(drop=True)
)

### 8.2.4. Rounding `float64` Column

In [13]:
infrabel_benchmark["regelmaat"] = infrabel_benchmark["regelmaat"].round(2)

### 8.2.5. Renaming Columns

In [17]:
infrabel_benchmark = infrabel_benchmark.rename(columns={
    "jaar" : "Year",
    "maand" : "Month",
    "regelmaat" : "Punctuality_Pct",
    "tel" : "Train_Count",
    "reg" : "Train_6Min_Late_Count",
    "min_rt" : "Delay_Minutes"
})

In [18]:
infrabel_benchmark.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype        
---  ------                 --------------  -----        
 0   Year                   77 non-null     int32        
 1   Month                  77 non-null     datetime64[s]
 2   Punctuality_Pct        77 non-null     float64      
 3   Train_Count            77 non-null     int64        
 4   Train_6Min_Late_Count  77 non-null     int64        
 5   Delay_Minutes          77 non-null     int64        
dtypes: datetime64[s](1), float64(1), int32(1), int64(3)
memory usage: 3.4 KB


## 8.3. Export to Gold Layer

The **Infrabel benchmark** is now ready to be exported to the `processed` directory as `infrabel_benchmark_prepared.parquet`.  

In [19]:
infrabel_benchmark.to_parquet(PATH_PROCESSED / "infrabel_benchmark_prepared.parquet")

In [20]:
df_to_verify = pd.read_parquet(PATH_PROCESSED / "infrabel_benchmark_prepared.parquet")
df_to_verify.head()

,Year,Month,Punctuality_Pct,Train_Count,Train_6Min_Late_Count,Delay_Minutes
0,2026,2026-05-01,92.39,109777,101421,184749
1,2026,2026-04-01,93.40,113704,106197,165290
2,2026,2026-03-01,93.05,112550,104728,181743
3,2026,2026-02-01,93.25,108663,101330,162212
4,2026,2026-01-01,92.39,107866,99662,178328


In [21]:
print(df_to_verify.shape)
df_to_verify.dtypes.to_dict()

(77, 6)


{'Year': dtype('int32'),
 'Month': dtype('<M8[ms]'),
 'Punctuality_Pct': dtype('float64'),
 'Train_Count': dtype('int64'),
 'Train_6Min_Late_Count': dtype('int64'),
 'Delay_Minutes': dtype('int64')}